# Data Cleaning

**Source (strict):** Kaggle Data Cleaning micro-course: https://www.kaggle.com/learn/data-cleaning

**Dataset:** any messy/dirty dataset (search 'dirty data csv' on Kaggle)

Attempt each question in its own code cell below. Add a short markdown note with your answer/finding after each.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv(r"C:\Users\shata\OneDrive\Desktop\dirty_cafe_sales.csv")
# df = pd.read_csv('your_dataset.csv')  # <-- point this at your downloaded dataset

### Q1. Report % of missing values per column, sorted descending.

In [2]:
df.isna().mean().mul(100).sort_values(ascending=False)

Location            32.65
Payment Method      25.79
Item                 3.33
Price Per Unit       1.79
Total Spent          1.73
Transaction Date     1.59
Quantity             1.38
Transaction ID       0.00
dtype: float64

### Q2. Drop columns where >50% of values are missing.

In [3]:
df = df.loc[:, df.isna().mean() <= 0.50]

### Q3. Fill missing numeric values with median; fill missing categorical values with mode.

In [4]:
# Numeric → median
df.fillna(df.select_dtypes(include='number').median(), inplace=True)

# Categorical → mode
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\shata\AppData\Local\Temp\ipykernel_15136\109952175.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
C:\Users\shata\AppData\Local\Temp\ipykernel_15136\109952175.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing '

### Q4. Find and remove exact duplicate rows; report how many were removed.

In [5]:
duplicates = df.duplicated().sum()
df = df.drop_duplicates()

print("Number of duplicate rows removed:", duplicates)

Number of duplicate rows removed: 0


### Q5. Strip whitespace and lowercase all string columns using `.apply()`.

In [6]:
df[df.select_dtypes(include='object').columns] = (
    df.select_dtypes(include='object')
      .apply(lambda col: col.str.strip().str.lower())
)

C:\Users\shata\AppData\Local\Temp\ipykernel_15136\2183359830.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include='object')
C:\Users\shata\AppData\Local\Temp\ipykernel_15136\2183359830.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.

### Q6. Merge inconsistent category spellings into one canonical value.

In [7]:
df['Transaction Date'].isna().sum()

np.int64(159)

### Q7. Convert mixed-format date strings to datetime64 using `pd.to_datetime(errors='coerce')`.

In [8]:
df_raw = pd.read_csv(r"C:\Users\shata\OneDrive\Desktop\dirty_cafe_sales.csv")  
bad_mask = pd.to_datetime(df_raw['Transaction Date'], errors='coerce').isna()
print("Problematic original values:", df_raw[bad_mask]['Transaction Date'].unique())

# Convert the column
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# Confirm dtype and check how many failed to parse
print("New dtype:", df['Transaction Date'].dtype)
print("Rows that failed to convert (now NaT):", df['Transaction Date'].isna().sum())

Problematic original values: <StringArray>
['ERROR', nan, 'UNKNOWN']
Length: 3, dtype: str
New dtype: datetime64[us]
Rows that failed to convert (now NaT): 460


### Q8. Detect outliers in one numeric column using the IQR method.

In [9]:
# Q8. Detect outliers using IQR method

col = 'Total Spent'

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")
print(f"Number of outliers in '{col}': {len(outliers)}")


TypeError: unsupported operand type(s) for -: 'str' and 'str'

### Q9. Fix a currency-formatted column (e.g. '$1,234.50') into a float column.

In [ ]:
df['Price Per Unit'] = (
    df['Price Per Unit']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')

print(df['Price Per Unit'].dtype)


float64


### Q10. Save the cleaned DataFrame to day2/cleaned_data.csv.

In [ ]:
# Q10. Save the cleaned DataFrame to day2/cleaned_data.csv

df.to_csv('cleaned_data.csv', index=False)